# OPTED Reverse Dictionary - Preprocessing Pipeline

Here we prepare the raw OPTED dictionary file for baseline and model experiments.

The raw file is kept unchanged. Cleaned files are saved separately under `data/processed/`.

## Step 1 - Load the raw dataset

Read the original CSV file. No changes are made to the raw dataset.

In [ ]:
from pathlib import Path
import hashlib
import re

import pandas as pd

raw_data_candidates = [
    Path("..") / "data" / "OPTED-Dictionary.csv",
    Path("data") / "OPTED-Dictionary.csv",
]

RAW_DATA_PATH = next(
    (path.resolve() for path in raw_data_candidates if path.exists()),
    None,
)

if RAW_DATA_PATH is None:
    raise FileNotFoundError("Could not find data/OPTED-Dictionary.csv")

ROOT_DIR = RAW_DATA_PATH.parent.parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

WORD_COL = "Word"
COUNT_COL = "Count"
POS_COL = "POS"
DEF_COL = "Definition"

raw_df = pd.read_csv(RAW_DATA_PATH, keep_default_na=False, dtype=str)

print(f"Loaded rows: {len(raw_df):,}")
display(raw_df.head())

## Step 2 - Validate required columns

Check that the expected OPTED columns are available before continuing.

In [ ]:
required_columns = {WORD_COL, COUNT_COL, POS_COL, DEF_COL}
missing_columns = required_columns - set(raw_df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("All required columns are present.")

## Step 3 - Clean text fields

Clean extra quotes, repeated spaces, and inconsistent casing.

In [ ]:
def normalize_spacing(value):
    text = str(value)
    text = text.replace('""""', '"')
    text = text.strip()
    text = text.strip('"')
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def basic_definition_clean(value):
    text = normalize_spacing(value).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


df = raw_df.copy()

df["word_original"] = df[WORD_COL].map(normalize_spacing)
df["definition_original"] = df[DEF_COL].map(normalize_spacing)
df["pos_original"] = df[POS_COL].map(normalize_spacing)
df["count_original"] = df[COUNT_COL].map(normalize_spacing)

df["word_norm"] = df["word_original"].str.lower()
df["definition_norm"] = df["definition_original"].str.lower()
df["definition_basic_clean"] = df["definition_original"].map(basic_definition_clean)

display(df[["word_original", "word_norm", "definition_original", "definition_basic_clean"]].head())

## Step 4 - Remove unusable rows

Remove rows without a usable word or definition. Spreadsheet artifacts such as `#NAME?` are also removed.

In [ ]:
before_rows = len(df)

invalid_word_values = {"", "#name?", "nan", "none"}

df = df[
    ~df["word_norm"].isin(invalid_word_values)
    & df["definition_norm"].ne("")
    & df["definition_basic_clean"].ne("")
].copy()

removed_rows = before_rows - len(df)

print(f"Rows removed because word/definition was blank after cleaning: {removed_rows:,}")
print(f"Rows remaining: {len(df):,}")

## Step 5 - Remove duplicate word-definition pairs

Remove repeated word-definition pairs. Different definitions for the same word are kept.

In [ ]:
before_dedup_rows = len(df)

df = (
    df.drop_duplicates(subset=["word_norm", "definition_norm"])
    .reset_index(drop=True)
    .copy()
)

duplicate_rows_removed = before_dedup_rows - len(df)

print(f"Duplicate word-definition rows removed: {duplicate_rows_removed:,}")
print(f"Rows remaining: {len(df):,}")

## Step 6 - Add quality and length features

Add word counts and simple flags for very short or very long definitions.

In [ ]:
word_pattern = re.compile(r"\b[\w'-]+\b")

df["definition_word_count"] = df["definition_original"].map(
    lambda text: len(word_pattern.findall(text))
)
df["clean_definition_word_count"] = df["definition_basic_clean"].map(
    lambda text: len(text.split())
)

df["is_short_definition"] = df["definition_word_count"] < 3
df["is_long_definition"] = df["definition_word_count"] > df["definition_word_count"].quantile(0.99)

display(
    df[
        [
            "word_original",
            "definition_original",
            "definition_word_count",
            "clean_definition_word_count",
            "is_short_definition",
            "is_long_definition",
        ]
    ].head()
)

## Step 7 - Preserve multiple senses

Keep each word-definition pair as its own sense. This matters because many words have multiple meanings.

In [ ]:
def make_entry_id(row):
    key = f"{row['word_norm']}\t{row['definition_norm']}"
    return hashlib.md5(key.encode("utf-8")).hexdigest()[:12]


df = df.sort_values(["word_norm", "definition_norm"]).reset_index(drop=True)
df["sense_number"] = df.groupby("word_norm").cumcount() + 1
df["entry_id"] = df.apply(make_entry_id, axis=1)

display(df[["entry_id", "word_original", "sense_number", "definition_original"]].head(10))

## Step 8 - Create train, validation, and test splits

Split by word so all definitions of the same word stay in the same split. This reduces leakage.

In [ ]:
RANDOM_SEED = 42
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10

unique_words = pd.Series(df["word_norm"].unique(), name="word_norm")
shuffled_words = unique_words.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

train_end = int(len(shuffled_words) * TRAIN_RATIO)
valid_end = int(len(shuffled_words) * (TRAIN_RATIO + VALID_RATIO))

train_words = set(shuffled_words.iloc[:train_end])
valid_words = set(shuffled_words.iloc[train_end:valid_end])
test_words = set(shuffled_words.iloc[valid_end:])

def assign_split(word):
    if word in train_words:
        return "train"
    if word in valid_words:
        return "valid"
    return "test"


df["split"] = df["word_norm"].map(assign_split)

split_summary = (
    df.groupby("split")
    .agg(
        rows=("entry_id", "size"),
        unique_words=("word_norm", "nunique"),
        avg_definition_words=("definition_word_count", "mean"),
    )
    .round(2)
    .reset_index()
)

display(split_summary)

## Step 9 - Select final columns

Keep the columns needed for modeling, evaluation, and later error analysis.

In [ ]:
processed_columns = [
    "entry_id",
    "split",
    "word_original",
    "word_norm",
    "sense_number",
    "definition_original",
    "definition_norm",
    "definition_basic_clean",
    "definition_word_count",
    "clean_definition_word_count",
    "is_short_definition",
    "is_long_definition",
    "count_original",
    "pos_original",
]

processed_df = df[processed_columns].copy()

display(processed_df.head())
print(f"Final processed rows: {len(processed_df):,}")

## Step 10 - Save processed files

Save the cleaned dataset and split files for the next notebooks.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_df.to_csv(PROCESSED_DIR / "opted_preprocessed.csv", index=False)

for split_name, split_df in processed_df.groupby("split"):
    split_df.to_csv(PROCESSED_DIR / f"opted_{split_name}.csv", index=False)

split_summary.to_csv(PROCESSED_DIR / "split_summary.csv", index=False)

print(f"Saved processed files to: {PROCESSED_DIR}")
print("Created:")
print("- opted_preprocessed.csv")
print("- opted_train.csv")
print("- opted_valid.csv")
print("- opted_test.csv")
print("- split_summary.csv")

## Preprocessing summary

Final output:

- Loads the raw English OPTED dataset.
- Checks that required columns exist.
- Cleans quotes, spaces, words, and definitions.
- Removes blank and duplicate word-definition rows.
- Keeps multiple meanings of the same word as separate senses.
- Adds definition length and quality flags.
- Creates leakage-aware train, validation, and test splits.
- Saves clean CSV files for baseline modeling.